## Prompt Engineering Langchain

In [ ]:
!pip install langchain_together

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.4/54.4 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 15.6 MB/s eta 0:00:00


In [ ]:
from langchain_together import ChatTogether

In [ ]:
llm = ChatTogether(
    together_api_key="aaf5debb6285cdd8e5342d52365486fefcdfadbf151a4e7f5b00e1f2a18a540c",
    model="meta-llama/Meta-Llama-3.1-8B-Instruct-Turbo",
)

In [ ]:
llm.invoke("hi")

AIMessage(content="How's it going? Is there something I can help you with or would you like to chat?", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 21, 'prompt_tokens': 11, 'total_tokens': 32, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_name': 'meta-llama/Llama-Vision-Free', 'system_fingerprint': None, 'finish_reason': 'stop', 'logprobs': None}, id='run-355a7e52-031f-4e22-9230-4c4efff83ae6-0', usage_metadata={'input_tokens': 11, 'output_tokens': 21, 'total_tokens': 32, 'input_token_details': {}, 'output_token_details': {}})

## Getting to know Prompt template

1. **`input_variables`**: Placeholder names (e.g., `['financial_concept']`) replaced with actual values in the template.  
2. **`template`**: String with placeholders (e.g., `demo_template`) where `input_variables` are inserted to create the final prompt.  
3. `StrOutputParser`: Formats the model's raw output into a desired structure, like a plain string, for easier use.

In [ ]:
from langchain import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

demo_template='''I want you to act as a acting financial advisor for people.
In an easy way, explain the basics of {financial_concept}.'''

prompt=PromptTemplate(
    input_variables=['financial_concept'],
    template=demo_template
    )

prompt.format(financial_concept='income tax')

'I want you to act as a acting financial advisor for people.\nIn an easy way, explain the basics of income tax.'

In [ ]:
chain1 = prompt | llm | StrOutputParser()

In [ ]:
chain1.invoke('GDP')

"As your financial advisor, I'd be happy to break down the basics of GDP in a simple way.\n\n**What is GDP?**\n\nGDP stands for Gross Domestic Product. It's a measure of a country's total economic output, which represents the value of all the goods and services produced within its borders over a specific period of time, usually a year.\n\n**How is GDP Calculated?**\n\nGDP is calculated by adding up the value of four main components:\n\n1. **Personal Consumption Expenditures (PCE)**: This includes the money spent by households on goods and services, such as food, clothing, housing, and entertainment.\n2. **Gross Investment**: This includes the money spent by businesses on new capital goods, such as buildings, equipment, and technology.\n3. **Government Spending**: This includes the money spent by the government on goods and services, such as military spending, infrastructure projects, and social programs.\n4. **Net Exports**: This includes the value of exports (goods and services sold t

## Prompts with multiple variables

In [ ]:
## Language Translation

from langchain import PromptTemplate

template='''In an easy way translate the following sentence '{sentence}' into {target_language}'''
language_prompt = PromptTemplate(
    input_variables=["sentence",'target_language'],
    template=template,
)
language_prompt.format(sentence="How are you",target_language='tamil')  # The format of the query

"In an easy way translate the following sentence 'How are you' into tamil"

In [ ]:
chain2 = language_prompt | llm | StrOutputParser()
chain2.invoke({'sentence':"Hello How are you",'target_language':'tamil'})

'Here is the translation:\n\n"வணக்கம் உங்கள் உடல் எப்படி இருக்கிறது?"\n\nThis can be shortened to a simpler translation:\n\n"வணக்கம் உங்க உடம்பு எப்படி இருக்கு?"\n\nHowever, if you want a more common way to say it, you can use:\n\n"வணக்கம் என்ன செய்திக்கு வந்தீர்கள்?"\n\nOr simply:\n\n"வணக்கம் என்ன என்று கேளுங்கள்?"\n\nNote: "வணக்கம்" is the formal way to say "hello", and the rest of the sentence is a more informal way to ask how someone is doing.'

In [ ]:
# Scenario based Example on Financial Domain with multiple inputs
demo_template = '''I want you to act as an acting financial advisor for people.
In an easy way, explain the basics of Simple Interest. Also, provide a simple, scenario-based example using a principal amount of {principal}, an annual interest rate of {interest_rate}%, and a time period of {time_period} years. Calculate the final amount.'''

# Create the PromptTemplate with three input variables
prompt = PromptTemplate(
    input_variables=['principal', 'interest_rate', 'time_period'],  # Three input variables
    template=demo_template
)

# Define the chain
chain1 = prompt | llm | StrOutputParser()

# Invoke the chain with specific values for principal, interest rate, and time period
chain1.invoke({'principal': '1000', 'interest_rate': '5', 'time_period': '3'})



"As your acting financial advisor, I'd be happy to explain the basics of simple interest in a way that's easy to understand.\n\n**What is Simple Interest?**\n\nSimple interest is a type of interest that's calculated as a percentage of the principal amount borrowed or invested, and it's paid only on the initial principal amount. The interest is not compounded, meaning it's not added to the principal amount to generate additional interest.\n\n**Key components of Simple Interest:**\n\n1. **Principal Amount (P):** The initial amount borrowed or invested.\n2. **Annual Interest Rate (R):** The percentage rate at which interest is charged or earned per year.\n3. **Time Period (T):** The number of years the money is borrowed or invested for.\n\n**Formula for Simple Interest:**\n\nSimple Interest (SI) = (P x R x T) / 100\n\n**Scenario-Based Example:**\n\nLet's use the following values:\n\n* Principal Amount (P) = $1000\n* Annual Interest Rate (R) = 5%\n* Time Period (T) = 3 years\n\n**Calculati

## Few Shot Prompting

- Few-shot prompting provides the model with a few input-output examples in the prompt for reference.  
- Diverse examples (varying in style, context, or complexity) help the model generalize better.  

In [ ]:
from langchain import PromptTemplate, FewShotPromptTemplate

# First, create the list of few shot examples.
examples = [
    {"word": "happy", "antonym": "sad"},
    {"word": "tall", "antonym": "short"},
]

# Next, we specify the template to format the examples we have provided.
# We use the `PromptTemplate` class for this.
example_formatter_template = """Word: {word}
Antonym: {antonym}
"""

example_prompt = PromptTemplate(
    input_variables=["word", "antonym"],
    template=example_formatter_template,
)

In [ ]:
# Finally, we create the `FewShotPromptTemplate` object.
few_shot_prompt = FewShotPromptTemplate(
    # These are the examples we want to insert into the prompt.
    examples=examples,
    # This is how we want to format the examples when we insert them into the prompt.
    example_prompt=example_prompt,
    # The prefix is some text that goes before the examples in the prompt.
    # Usually, this consists of intructions.
    prefix="Give the antonym of every input\n",
    # The suffix is some text that goes after the examples in the prompt.
    # Usually, this is where the user input will go
    suffix="Word: {input}\nAntonym: ",
    # The input variables are the variables that the overall prompt expects.
    input_variables=["input"],
    # The example_separator is the string we will use to join the prefix, examples, and suffix together with.
    example_separator="\n",
)

In [ ]:
print(few_shot_prompt.format(input='big'))

Give the antonym of every input

Word: happy
Antonym: sad

Word: tall
Antonym: short

Word: big
Antonym: 


In [ ]:
#chain=LLMChain(llm=llm,prompt=few_shot_prompt)
chain = few_shot_prompt | llm | StrOutputParser()
chain.invoke({'input':"big"})

'The antonym of "big" is "small".'

The discrepancy occurs because the model inconsistently uses "large" as the antonym for both "small" and "tiny" instead of "big" or "huge" suggesting a need for more diverse examples to improve accuracy.

In [ ]:
#chain=LLMChain(llm=llm,prompt=few_shot_prompt)
chain = few_shot_prompt | llm | StrOutputParser()
chain.invoke({'input':"small"})

'The antonym of "small" is "large".'

In [ ]:
#chain=LLMChain(llm=llm,prompt=few_shot_prompt)
chain = few_shot_prompt | llm | StrOutputParser()
chain.invoke({'input':"tiny"})

'The antonym of "tiny" is "large".'

In [ ]:
#chain=LLMChain(llm=llm,prompt=few_shot_prompt)
chain = few_shot_prompt | llm | StrOutputParser()
chain.invoke({'input':"miniscule"})

'The antonym of "miniscule" would be "enormous".'

# **Use case Based Example: Evaluation of Answer scripts**

**Zero Shot Prompt**

Zero-shot prompting involves asking the model to perform a task without providing any examples, relying solely on its pre-trained knowledge and the given instruction.

In [ ]:
# Define the prompt template
zero_shot_prompt = PromptTemplate(
    template="""
    You are an expert in evaluating student answers for Operating Systems (OS) questions.
    Evaluate the following answer based on the question and provide a grade between 0 and 5, where:


    Question: {question}
    Answer: {answer}

    Grade(0-5 marks only):
    """,
    input_variables=["question", "answer"]
)

# Example input
question = "What is a deadlock in operating systems?"
answer = "A deadlock is when two processes are waiting for each other to finish, but neither can proceed."


# Create the chain
chain = zero_shot_prompt | llm | StrOutputParser()

# Pass the input to the chain
result = chain.invoke({"question": question, "answer": answer})

# Print the result
print(result)

Grade: 2

The answer is partially correct but lacks detail and clarity. A deadlock is indeed a situation where two or more processes are blocked indefinitely, waiting for each other to release resources. However, the answer does not mention the key characteristics of a deadlock, such as the circular wait condition, mutual exclusion, and the hold and wait condition. It also simplifies the concept by stating that two processes are waiting for each other to finish, which is not entirely accurate. A more comprehensive answer would provide a clearer explanation of the deadlock concept.


**Few Shot Prompting**

In this case, we used two diverse examples (2-shot) with varying scores of 1 and 4 to improve its performance.

In [ ]:
# Define the few-shot examples
examples = [
    {
        "question": "What is virtual memory?",
        "answer": "Virtual memory is a type of memory that is used to store data temporarily.",
        "grade": 1,

    },
    {
        "question": "What is a process in an operating system?",
        "answer": "A process is a program in execution. It includes the program code, stack, data, and heap.",
        "grade": 4,
    }
]

# Define the example formatter template
example_formatter_template = """
Question: {question}
Answer: {answer}
Grade: {grade}
"""

example_prompt = PromptTemplate(
    input_variables=["question", "answer", "grade"],
    template=example_formatter_template,
)

# Define the few-shot prompt template
few_shot_prompt = FewShotPromptTemplate(
    examples=examples,
    example_prompt=example_prompt,
    prefix="""
    You are an expert in evaluating student answers for Operating Systems (OS) questions.
    Evaluate the following answer based on the question and provide a grade between 0 and 5, where:

    """,
    suffix="""
    Question: {question}
    Answer: {answer}

    Grade(0-5 marks only):
    """,
    input_variables=["question", "answer"],
    example_separator="\n\n",
)

# Example input
question = "What is a deadlock in operating systems?"
answer = "A deadlock is when two processes are waiting for each other to finish, but neither can proceed."


# Create the chain
chain = few_shot_prompt | llm | StrOutputParser()

# Pass the input to the chain
result = chain.invoke({"question": question, "answer": answer})

# Print the result
print(result)

Based on the answer, I would give a grade of 3 out of 5.

The answer correctly identifies a deadlock as a situation where two processes are waiting for each other to finish, but it lacks a crucial detail: the necessary conditions for a deadlock to occur (e.g., mutual exclusion, hold and wait, no preemption, and circular wait). However, it does provide a clear and concise definition of a deadlock, which is a good starting point.


**Few Shot Prompting(another example)**

In [ ]:
# Define the few-shot examples
examples = [
    {
        "question": "What is virtual memory?",
        "answer": "Virtual memory is a type of memory that is used to store data temporarily.",
        "grade": 1,

    },
    {
        "question": "What is a process in an operating system?",
        "answer": "A process is a program in execution. It includes the program code, stack, data, and heap.",
        "grade": 4,
    }
]

# Define the example formatter template
example_formatter_template = """
Question: {question}
Answer: {answer}
Grade: {grade}
"""

example_prompt = PromptTemplate(
    input_variables=["question", "answer", "grade"],
    template=example_formatter_template,
)

# Define the few-shot prompt template
few_shot_prompt = FewShotPromptTemplate(
    examples=examples,
    example_prompt=example_prompt,
    prefix="""
    You are an expert in evaluating student answers for Operating Systems (OS) questions.
    Evaluate the following answer based on the question and provide a grade between 0 and 5, where:

    """,
    suffix="""
    Question: {question}
    Answer: {answer}

    Grade(0-5 marks only):
    """,
    input_variables=["question", "answer"],
    example_separator="\n\n",
)

question = """
Explain the concept of virtual memory in operating systems. Include details about how it works, its benefits, and its impact on system performance.
"""

answer = """
Virtual memory is a memory management technique used by operating systems to allow processes to use more memory than is physically available.
It works by dividing memory into fixed-size blocks called pages, which are stored in RAM or on disk. When a process needs a page that is not in RAM,
a page fault occurs, and the OS retrieves the page from disk. This allows multiple processes to share the same physical memory efficiently.
The benefits of virtual memory include increased system stability, as processes are isolated from each other, and the ability to run larger applications
than the physical memory would normally allow. However, excessive paging can lead to performance degradation, known as thrashing.
"""

# Create the chain
chain = few_shot_prompt | llm | StrOutputParser()

# Pass the input to the chain
result = chain.invoke({"question": question, "answer": answer})

# Print the result
print(result)

Grade: 5

The answer provides a clear and comprehensive explanation of virtual memory, including its concept, working, benefits, and impact on system performance. It accurately describes the page-based memory management technique, the page fault process, and the benefits of increased system stability and the ability to run larger applications. Additionally, it mentions the potential drawback of excessive paging leading to performance degradation. The answer demonstrates a thorough understanding of the topic and is well-structured and easy to follow.


**Chain of Thought Prompting**

Chain of thought prompting encourages the model to generate intermediate reasoning steps before arriving at the final answer, improving its ability to solve complex problems.

In [ ]:
# Define the prompt template with Chain-of-Thought reasoning
cot_prompt = PromptTemplate(
    template="""
    You are an expert in evaluating student answers for Operating Systems (OS) questions.
    Your task is to evaluate the following answer based on the question and provide a grade between 0 and 5:


    Follow these steps to evaluate the answer:
    1. **Identify Key Concepts**: Check if the answer includes the key concepts related to the question.
    2. **Check for Accuracy**: Verify if the information provided is accurate and relevant to the question.
    3. **Evaluate Completeness**: Assess whether the answer covers all aspects of the question or leaves out important details.
    4. **Assess Clarity**: Determine if the answer is clear, well-structured, and easy to understand.
    5. **Assign a Grade**: Based on the above steps, assign a grade between 0 and 5.

    Question: {question}
    Answer: {answer}

    Provide the grade and a detailed explanation for the evaluation, including the steps you followed.
    Grade(0-5 marks only):
    """,
    input_variables=["question", "answer"]
)

# Example input (longer question and answer)
question = """
Explain the concept of virtual memory in operating systems. Include details about how it works, its benefits, and its impact on system performance.
"""

answer = """
Virtual memory is a memory management technique used by operating systems to allow processes to use more memory than is physically available.
It works by dividing memory into fixed-size blocks called pages, which are stored in RAM or on disk. When a process needs a page that is not in RAM,
a page fault occurs, and the OS retrieves the page from disk. This allows multiple processes to share the same physical memory efficiently.
The benefits of virtual memory include increased system stability, as processes are isolated from each other, and the ability to run larger applications
than the physical memory would normally allow. However, excessive paging can lead to performance degradation, known as thrashing.
"""


# Create the chain
chain = cot_prompt | llm | StrOutputParser()

# Pass the input to the chain
result = chain.invoke({"question": question, "answer": answer})

# Print the result
print(result)

Grade: 4

**Detailed Explanation:**

**Step 1: Identify Key Concepts**
The answer includes the key concepts related to the question, such as:
- Virtual memory as a memory management technique
- Division of memory into fixed-size blocks called pages
- Storage of pages in RAM or on disk
- Page faults and retrieval of pages from disk
- Benefits of virtual memory, including increased system stability and ability to run larger applications
- Impact of excessive paging on system performance (thrashing)

**Step 2: Check for Accuracy**
The information provided is accurate and relevant to the question. The answer correctly explains how virtual memory works, its benefits, and its impact on system performance.

**Step 3: Evaluate Completeness**
The answer covers most aspects of the question, but it could be more comprehensive. For example, it does not mention the role of the page table in managing page faults or the concept of swapping pages between RAM and disk. However, it does provide a clear 

**Meta Prompting**

Meta prompting focuses on the structure and design of the prompt itself, rather than relying on specific examples, to guide the model in generating or refining its own prompts, enhancing its ability to tackle tasks more effectively and adaptively.

In [ ]:

meta_prompt = PromptTemplate(
    template="""
    You are an expert in evaluating student answers for Operating Systems (OS) questions.
    Your task is to evaluate the following answer based on the question and provide a grade between 0 and 5:


    **Evaluation Criteria**:
    The answer should follow this structure:
    1. **Introduction**: A brief overview of the topic.
    2. **Key Concepts**: Explanation of the main concepts related to the question.
    3. **Explanation**: Detailed explanation of how the concepts work or their significance.
    4. **Conclusion**: A summary or final thoughts on the topic.

    Evaluate the answer based on how well it adheres to this structure. Check for:
    - Presence of all required sections (Introduction, Key Concepts, Explanation, Conclusion).
    - Logical flow and coherence between sections.
    - Clarity and completeness of each section.

    Question: {question}
    Answer: {answer}

    Provide the grade and a detailed explanation for the evaluation, including how well the answer follows the required structure.
    Grade(0-5 marks only):
    """,
    input_variables=["question", "answer"]
)

# Example input (longer question and answer)
question = """
Explain the concept of virtual memory in operating systems. Include details about how it works, its benefits, and its impact on system performance.
"""

answer = """
Virtual memory is a memory management technique used by operating systems to allow processes to use more memory than is physically available.
It works by dividing memory into fixed-size blocks called pages, which are stored in RAM or on disk. When a process needs a page that is not in RAM,
a page fault occurs, and the OS retrieves the page from disk. This allows multiple processes to share the same physical memory efficiently.
The benefits of virtual memory include increased system stability, as processes are isolated from each other, and the ability to run larger applications
than the physical memory would normally allow. However, excessive paging can lead to performance degradation, known as thrashing.
"""

chain = meta_prompt | llm | StrOutputParser()

# Pass the input to the chain
result = chain.invoke({"question": question, "answer": answer})

# Print the result
print(result)

Grade: 4

The answer provides a good explanation of the concept of virtual memory, but it lacks a clear conclusion and some details that would make it more comprehensive. Here's a breakdown of the evaluation:

**Introduction (1/1)**: The answer starts with a brief overview of virtual memory, which is a good introduction to the topic. However, it could be more concise and clear.

**Key Concepts (2/2)**: The answer explains the main concepts related to virtual memory, including how it works, page faults, and the division of memory into fixed-size blocks. However, it could provide more details about the page replacement algorithms and the role of the page table.

**Explanation (3/4)**: The answer provides a good explanation of how virtual memory works, including the process of page faults and the benefits of virtual memory. However, it lacks some details about the impact of virtual memory on system performance, such as the effects of thrashing and the role of the operating system in manag

## Example Selectors

*   This method of example selection chooses the best example from a predefined list by comparing the length of the input word to the length of words in the examples, selecting the one with the smallest length difference.
*   It is useful for scenarios where the relevance of examples is influenced by the similarity in length, such as in text-based tasks where input and example size alignment matters.



In [ ]:
from langchain_core.example_selectors.base import BaseExampleSelector

In [ ]:
class CustomExampleSelector(BaseExampleSelector):
    def __init__(self, examples):
        self.examples = examples

    #overriding
    def add_example(self, example):
        self.examples.append(example)

    #overriding
    def select_examples(self, input_variables):
        #input variable represents the initial state of the chain - dict
        # This assumes knowledge that part of the input will be a 'text' key
        new_word = input_variables["input"]
        new_word_length = len(new_word)

        # Initialize variables to store the best match and its length difference
        best_match = None
        smallest_diff = float("inf")

        # Iterate through each example
        for example in self.examples:
            # Calculate the length difference with the first word of the example
            current_diff = abs(len(example["input"]) - new_word_length)

            # Update the best match if the current one is closer in length
            if current_diff < smallest_diff:
                smallest_diff = current_diff
                best_match = example

        return [best_match]

In [ ]:
examples = [
    {"input": "hi", "output": "ciao"},
    {"input": "bye", "output": "arrivaderci"},
    {"input": "soccer", "output": "calcio"},
]

In [ ]:
example_selector = CustomExampleSelector(examples)

In [ ]:
example_selector.select_examples({"input": "okay"})

[{'input': 'bye', 'output': 'arrivaderci'}]

In [ ]:
example_selector.add_example({"input": "hand", "output": "mano"})

In [ ]:
example_selector.select_examples({"input": "okay"})

[{'input': 'hand', 'output': 'mano'}]

In [ ]:
example_prompt = PromptTemplate.from_template("Input: {input} -> Output: {output}")

In [ ]:
prompt = FewShotPromptTemplate(
    example_selector=example_selector,
    example_prompt=example_prompt,
    suffix="Input: {input} -> Output:",
    prefix="Translate the following words from English to Italain:",
    input_variables=["input"],
)

In [ ]:
print(prompt.format(input="word"))

Translate the following words from English to Italain:

Input: hand -> Output: mano

Input: word -> Output:


In [ ]:
chain = prompt | llm | StrOutputParser()

In [ ]:
chain.invoke({"input":"word"})

'The translation of "word" from English to Italian is "parola".'

# **Using MMR Example Selector**

1.   The **Max Marginal Relevance ExampleSelector (MMR)** adds diversity by ensuring selected examples are not only similar to the input but also distinct from each other.
2. This approach is more efficient than simply selecting the top similar examples, as it avoids redundancy and enriches the prompt with diverse information.
3. It is particularly useful in few-shot learning to avoid redundant examples and provide a broader context for the model to generalize effectively.
2.   The methodology involves using **FAISS (Facebook AI Similarity Search)** to compute **cosine similarity** between embeddings. Initially, the most similar example to the input is selected. For subsequent examples, MMR balances similarity to the input and dissimilarity to already selected examples, using a combination of cosine similarity scores. This ensures the final set of examples is both relevant and diverse, enhancing the quality of the prompt.




In [ ]:
!pip install sentence-transformers faiss-cpu langchain_community

from langchain.embeddings import HuggingFaceEmbeddings
from langchain.vectorstores import FAISS
from langchain.prompts.example_selector import MaxMarginalRelevanceExampleSelector
from langchain.prompts import PromptTemplate, FewShotPromptTemplate

In [ ]:


# Define the examples (LLM-related)
examples = [
    {"input": "BERT", "output": "Text understanding"},
    {"input": "GPT-3", "output": "Text generation"},
    {"input": "T5", "output": "Text-to-text tasks"},
    {"input": "RoBERTa", "output": "Robust text understanding"},
    {"input": "XLM-R", "output": "Multilingual text understanding"},
    {"input": "DALL-E", "output": "Image generation"},
    {"input": "CLIP", "output": "Image-text understanding"},
    {"input": "Stable Diffusion", "output": "Image generation"},
    {"input": "AlphaTensor", "output": "Math problem solving"},
    {"input": "GPT-4", "output": "Math reasoning"},
    {"input": "CodeBERT", "output": "Code understanding"},
    {"input": "Whisper", "output": "Speech-to-text"},
]

# Define the example prompt
example_prompt = PromptTemplate(
    input_variables=["input", "output"],
    template="Input: {input}\nOutput: {output}",
)

# Initialize Sentence Transformers embeddings
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")  # Lightweight and efficient model

# Create the MaxMarginalRelevanceExampleSelector
example_selector = MaxMarginalRelevanceExampleSelector.from_examples(
    # The list of examples available to select from.
    examples,
    # The embedding class used to produce embeddings.
    embeddings,
    # The VectorStore class used to store embeddings and perform similarity search.
    FAISS,
    # The number of examples to produce.
    k=2,
)

# Create the FewShotPromptTemplate
similar_prompt = FewShotPromptTemplate(
    # We provide an ExampleSelector instead of examples.
    example_selector=example_selector,
    example_prompt=example_prompt,
    prefix="Give the functionality of the following model:",
    suffix="Input: {model}\nOutput:",
    input_variables=["model"],
)

# Test the prompt with a query
query = "ViT (Vision Transformer)"
formatted_prompt = similar_prompt.format(model=query)

# Print the formatted prompt
print(formatted_prompt)

Give the functionality of the following model:

Input: DALL-E
Output: Image generation

Input: RoBERTa
Output: Robust text understanding

Input: ViT (Vision Transformer)
Output:


In [ ]:
chain = similar_prompt| llm | StrOutputParser()

# Test the chain with a query
query = "ViT (Vision Transformer)"
result = chain.invoke({"model": query})

# Print the result
print(result)

Based on the input models, here's a possible functionality for the output model:

Input: ViT (Vision Transformer)
Output: Image classification or Image feature extraction

The Vision Transformer (ViT) is a type of transformer-based model that is primarily used for image classification tasks. It takes in a sequence of image patches and outputs a classification label or a set of features that can be used for further processing.

Some possible functionalities of the output model could be:

1. **Image classification**: Given an input image, the model can classify it into one of the predefined categories (e.g., animals, vehicles, buildings, etc.).
2. **Image feature extraction**: The model can extract features from an input image that can be used for further processing, such as object detection, segmentation, or image retrieval.
3. **Image captioning**: The model can generate a caption or a description for an input image.
4. **Image generation**: Although less common, some variants of ViT c

## Partial Prompts

In [ ]:
from datetime import datetime

def _get_datetime():
    now = datetime.now()
    return now.strftime("%m/%d/%Y")

In [ ]:
prompt = PromptTemplate(
    template="Tell me a few of the {adjective} event happened on the same day of the date: {date} in the past",
    input_variables=["adjective", "date"],
)

In [ ]:
partial_prompt = prompt.partial(date=_get_datetime)

In [ ]:
print(partial_prompt.format(adjective="important"))

Tell me a few of the important event happened on the same day of the date: 01/13/2025 in the past


In [ ]:
chain = partial_prompt | llm | StrOutputParser()

In [ ]:
chain.invoke({"adjective":"important"})

'I\'m not aware of any specific significant events that occurred on October 6th, 2024, as my knowledge cutoff is December 2023. I can, however, provide you with some notable events that occurred on October 6th in previous years.\n\nHere are a few important events that occurred on October 6th in the past:\n\n1. 1869 - Cyrus West Field laid the first transatlantic telegraph cable, which successfully carried the message "Europe to America" from Valentia Island, Ireland to Trinity Bay, Newfoundland.\n\n2. 1943 - The Battle of San Pietro, a 12-day battle during World War II, ended with Allied forces taking control of San Pietro, Italy.\n\n3. 1949 - The People\'s Republic of China was proclaimed by Mao Zedong, marking the beginning of the Communist era in China.\n\n4. 1962 - The first Walgreens store was opened in Chicago, Illinois.\n\n5. 1973 - The US Congress passed the Indian Self-Determination and Education Assistance Act, which allowed Native American tribes to take greater control of t

# **Output Parser(*comma separated values*)**

The **Output Parser** converts structured data, such as lists or JSON, into **comma-separated values (CSV)**, making it easier to store, share, or analyze the extracted information in a tabular format.

In [ ]:
from langchain.prompts import PromptTemplate, FewShotPromptTemplate
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.vectorstores import FAISS
from langchain.output_parsers import CommaSeparatedListOutputParser
from langchain.schema.runnable import RunnablePassthrough
from langchain.schema import StrOutputParser

# Define the examples (LLM-related) with reversed input and output
examples = [
    {"input": "Text understanding", "output": "BERT"},
    {"input": "Text generation", "output": "GPT-3"},
    {"input": "Text-to-text tasks", "output": "T5"},
    {"input": "Robust text understanding", "output": "RoBERTa"},
    {"input": "Multilingual text understanding", "output": "XLM-R"},
    {"input": "Image generation", "output": "DALL-E"},
    {"input": "Image-text understanding", "output": "CLIP"},
    {"input": "Image generation", "output": "Stable Diffusion"},
    {"input": "Math problem solving", "output": "AlphaTensor"},
    {"input": "Math reasoning", "output": "GPT-4"},
    {"input": "Code understanding", "output": "CodeBERT"},
    {"input": "Speech-to-text", "output": "Whisper"},
]

# Define the example prompt
example_prompt = PromptTemplate(
    input_variables=["input", "output"],
    template="Input: {input}\nOutput: {output}",
)

# Initialize Sentence Transformers embeddings
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")  # Lightweight and efficient model

# Create the MaxMarginalRelevanceExampleSelector
example_selector = MaxMarginalRelevanceExampleSelector.from_examples(
    # The list of examples available to select from.
    examples,
    # The embedding class used to produce embeddings.
    embeddings,
    # The VectorStore class used to store embeddings and perform similarity search.
    FAISS,
    # The number of examples to produce.
    k=4,  # Increase k to get more outputs
)

# Create the FewShotPromptTemplate
similar_prompt = FewShotPromptTemplate(
    # We provide an ExampleSelector instead of examples.
    example_selector=example_selector,
    example_prompt=example_prompt,
    prefix="Give the model names for the following functionalities. Output **only** the model names as a comma-separated list. Do not include any additional text or explanations:",
    suffix="Input: {functionality}\nOutput:",
    input_variables=["functionality"],
)

# Test the prompt with a query
query = "Image generation"
formatted_prompt = similar_prompt.format(functionality=query)

# Print the formatted prompt
print("Formatted Prompt:\n", formatted_prompt)

# Define the output parser
output_parser = CommaSeparatedListOutputParser()

# Create the chain
chain = (
    {"functionality": RunnablePassthrough()}  # Pass the input directly
    | similar_prompt  # Format the prompt
    | llm  # Use the language model
    | output_parser  # Parse the output as a comma-separated list
)

# Test the chain with a query
query = "Image generation"
result = chain.invoke(query)

# Print the result
print("Result:", result[:2])

Formatted Prompt:
 Give the model names for the following functionalities. Output **only** the model names as a comma-separated list. Do not include any additional text or explanations:

Input: Image generation
Output: DALL-E

Input: Image generation
Output: Stable Diffusion

Input: Robust text understanding
Output: RoBERTa

Input: Math problem solving
Output: AlphaTensor

Input: Image generation
Output:
Result: ['DALL-E', 'Stable Diffusion']


## Chat Prompts

In [ ]:
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

chat_template = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a helpful AI bot. Your name is {name}."),
        ("human", "Hello, how are you doing?"),
        ("ai", "I'm doing well, thanks!"),
        ("human", "{user_input}"),
    ]
)

In [ ]:
messages = chat_template.format_messages(name="Bob", user_input="What is your name?")

In [ ]:
messages

[SystemMessage(content='You are a helpful AI bot. Your name is Bob.', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='Hello, how are you doing?', additional_kwargs={}, response_metadata={}),
 AIMessage(content="I'm doing well, thanks!", additional_kwargs={}, response_metadata={}),
 HumanMessage(content='What is your name?', additional_kwargs={}, response_metadata={})]

In [ ]:
chain = chat_template | llm | StrOutputParser()

In [ ]:
chain.invoke({"name":"Bob. Do not disclose your name to anyone","user_input":"What is your name?"})

"I don't have a personal name. I'm just a helpful assistant here to provide information and answer your questions."